# XGBoost Model

- Eric Hernandez

In [ ]:
# Fix import path (add parent directory)
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
print(f"Added to path: {project_root}")


In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import confusion_matrix
from xgboost import XGBClassifier


In [ ]:
def load_subject_pickle(pkl_path):
    """Load a subject's pickle file containing raw sensor data."""
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')
    return data


In [ ]:
def create_windows(signal, window_size, step_size):
    return [
        signal[i:i + window_size]
        # #TODO: Handle last window if it doesn't fit perfectly (+1 to include last point)
        for i in range(0, len(signal) - window_size, step_size)
    ]


In [ ]:
def get_window_labels(labels, window_size, step_size):
    y = []

    for i in range(0, len(labels) - window_size, step_size):
        window = labels[i:i + window_size]
        window = window[window != 0]  # remove transient

        if len(window) == 0:
            continue

        majority = np.bincount(window).argmax()

        if majority == 2:
            y.append(1)  # Stress
        elif majority in [1, 3]:
            y.append(0)  # Non-stress
        else:
            continue

    return y


In [ ]:
def extract_features(window):
    return [
        np.mean(window),
        np.std(window),
        np.min(window),
        np.max(window),
        np.percentile(window, 25),
        np.percentile(window, 75),
        np.median(window)
    ]


In [ ]:
def process_subject(subject_path, window_size, step_size):
    subject_data = load_subject_pickle(subject_path)

    chest = subject_data["signal"]["chest"]
    labels = subject_data["label"]

    ecg = chest["ECG"]
    eda = chest["EDA"]
    resp = chest["Resp"]
    temp = chest["Temp"]
    emg = chest["EMG"]
    acc = chest["ACC"]  # shape: (n_samples, 3)

    # Handle ACC channels separately
    acc_x = acc[:, 0]
    acc_y = acc[:, 1]
    acc_z = acc[:, 2]

    ecg_w = create_windows(ecg, window_size, step_size)
    eda_w = create_windows(eda, window_size, step_size)
    resp_w = create_windows(resp, window_size, step_size)
    temp_w = create_windows(temp, window_size, step_size)
    emg_w = create_windows(emg, window_size, step_size)

    # Create windows for each ACC channel
    acc_x_w = create_windows(acc_x, window_size, step_size)
    acc_y_w = create_windows(acc_y, window_size, step_size)
    acc_z_w = create_windows(acc_z, window_size, step_size)

    y = get_window_labels(labels, window_size, step_size)

    X = []
    valid_len = min(len(y), len(ecg_w))

    for i in range(valid_len):
        features = []
        
        features += extract_features(ecg_w[i])
        features += extract_features(eda_w[i])
        features += extract_features(resp_w[i])
        features += extract_features(temp_w[i])
        features += extract_features(emg_w[i])

        features += extract_features(acc_x_w[i])
        features += extract_features(acc_y_w[i])
        features += extract_features(acc_z_w[i])

        X.append(features)

    return X, y[:valid_len]


In [ ]:
DATASET_PATH = '../data/WESAD'

FS = 700
WINDOW_SEC = 60
WINDOW_SIZE = FS * WINDOW_SEC
STEP_SIZE = WINDOW_SIZE // 2  # 50% overlap

all_X = []
all_y = []
all_subject_ids = []

subjects = sorted([
    d for d in os.listdir(DATASET_PATH)
    if d.startswith("S")
])

for subject in subjects:
    subject_path = os.path.join(DATASET_PATH, subject, f"{subject}.pkl")

    if not os.path.exists(subject_path):
        continue

    print(f"Processing {subject}...")

    X, y = process_subject(subject_path, WINDOW_SIZE, STEP_SIZE)

    all_X.extend(X)
    all_y.extend(y)
    all_subject_ids.extend([subject] * len(y))


In [ ]:
columns = [
    "ECG_mean", "ECG_std", "ECG_min", "ECG_max", "ECG_25th", "ECG_75th", "ECG_median",
    "EDA_mean", "EDA_std", "EDA_min", "EDA_max", "EDA_25th", "EDA_75th", "EDA_median",
    "RESP_mean", "RESP_std", "RESP_min", "RESP_max", "RESP_25th", "RESP_75th", "RESP_median",
    "TEMP_mean", "TEMP_std", "TEMP_min", "TEMP_max", "TEMP_25th", "TEMP_75th", "TEMP_median",
    "EMG_mean", "EMG_std", "EMG_min", "EMG_max", "EMG_25th", "EMG_75th", "EMG_median",
    "ACC_X_mean", "ACC_X_std", "ACC_X_min", "ACC_X_max", "ACC_X_25th", "ACC_X_75th", "ACC_X_median",
    "ACC_Y_mean", "ACC_Y_std", "ACC_Y_min", "ACC_Y_max", "ACC_Y_25th", "ACC_Y_75th", "ACC_Y_median",
    "ACC_Z_mean", "ACC_Z_std", "ACC_Z_min", "ACC_Z_max", "ACC_Z_25th", "ACC_Z_75th", "ACC_Z_median"
]

df = pd.DataFrame(all_X, columns=columns)
df["label"] = all_y
df["subject"] = all_subject_ids

df.describe()


In [ ]:
test_subjects = ["S15", "S16", "S17"]
train_subjects = [s for s in subjects if s not in test_subjects]

train_df = df[df["subject"].isin(train_subjects)]
test_df = df[df["subject"].isin(test_subjects)]

X_train = train_df.drop(["label", "subject"], axis=1)
y_train = train_df["label"]

X_test = test_df.drop(["label", "subject"], axis=1)
y_test = test_df["label"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
model = XGBClassifier(
    eval_metric='logloss',
    random_state=42,
)

parameters_xgb = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 8],
    'learning_rate': [0.05, 0.1, 0.2]
}

model_optimized = RandomizedSearchCV(
    model,
    parameters_xgb,
    n_iter=5,
    cv=3,
    random_state=42
)

model_optimized.fit(
    X_train_scaled, 
    y_train
)



In [ ]:
predictions = model_optimized.predict(X_test_scaled)

print("Model Summary: \n")
print(classification_report(y_test, predictions))


In [ ]:
# Visualize feature importance
plt.figure(figsize=(10, 14))
plt.barh(columns, model_optimized.best_estimator_.feature_importances_)
plt.xlabel("Feature Importance")
plt.title("XGBoost Feature Importance")
plt.show()


In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, predictions)
labels = ['True Negative', 'False Positive', 'False Negative', 'True Positive']
grouped_labels = [f"{label}\nCount: {value}" for label, value in zip(labels, cm.ravel())]
grouped_labels = np.asarray(grouped_labels).reshape(2, 2)

# Unpack the confusion matrix values
tn, fp, fn, tp = cm.ravel()

# Print each value with its label
print(f"True Negative (TN): {tn}")
print(f"False Positive (FP): {fp}")
print(f"False Negative (FN): {fn}")
print(f"True Positive (TP): {tp}")

# Plot heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=grouped_labels, fmt='', cmap='Blues', cbar=False,
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])

plt.title('Confusion Matrix - XGBoost Optimized')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()



In [ ]:
plt.figure(figsize=(8, 6))

# XGBoost Optimized
probs_xgb = model_optimized.predict_proba(X_test_scaled)[:, 1]
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, probs_xgb)
roc_auc_xgb = auc(fpr_xgb, tpr_xgb)
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC = {roc_auc_xgb:.2f})")

# Plot formatting
plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison of Models')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()
